# Week 2B · Camera and telescope → science

Monday's (Sep 28) "Design your own survey" worksheet handed you a single-visit depth,
r ≈ 24.7, as a given. In Notebook A the simulated survey's median r-band visit reached only
24.06. This notebook builds that depth from the hardware up: mirrors, lenses, filters, detector,
atmosphere, and the night sky. It then follows the camera through to science: seeing and weak
lensing, and the layout of the focal plane. It goes with Aaron Roodman's lecture on Wednesday
(Sep 30), and it compares with the first on-sky numbers from Phil Marshall's Sep 23 overview.

**How to use it.** In Colab, choose *Runtime → Run all*, then move the sliders. Every slider
wraps a plain function you can call directly. Use the direct call for your hand-in figure,
because sliders don't show up in PDFs.

**Core path vs. go deeper.** The core path takes about 30–45 minutes if you don't stop to answer
every prompt. The bold prompts are for thinking and discussion (in class and on Slack); only the
hand-in is turned in. The *Go deeper* boxes are optional.

*Data note: the throughput curves are the Rubin project's engineering model of the as-built
hardware (public, from GitHub), and the survey numbers are from public simulations. None of this
is measured Rubin data.*

In [ ]:
# Setup. Everything here ships with Colab.
import os
from pathlib import Path
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/rhw/phys367/main/week2/data/"
DATA_DIR = Path("data")
DATA_FILES = {
    "throughputs": "throughputs.csv",
    "darksky": "darksky.csv",
    "m5_reference": "m5_reference.csv",
}

def _fetch(name):
    """Local path of a data file; downloads it from GitHub once if it isn't in data/."""
    fname = DATA_FILES.get(name, name)
    path = DATA_DIR / fname
    if not path.exists():
        DATA_DIR.mkdir(exist_ok=True)
        print(f"downloading {fname} ...")
        part = path.with_suffix(path.suffix + ".part")   # don't trust a half-finished download
        urllib.request.urlretrieve(DATA_URL + fname, part)
        part.rename(path)
    return path

def load(name):
    """Load a data file by short name ('throughputs', 'darksky', 'm5_reference') or file name.

    Uses the local data/ directory if the file is there; otherwise downloads it from
    GitHub once and caches it in data/. Lines starting with '#' are the file's provenance
    header; read them with header(name).
    """
    path = _fetch(name)
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, comment="#")

def header(name):
    """The provenance header (the '#' lines) of a data file, as one string."""
    with open(_fetch(name)) as f:
        return "".join(line for line in f if line.startswith("#"))

from ipywidgets import interact, interact_manual

def sliders(func, **controls):
    """Attach sliders and menus to func, like ipywidgets.interact.

    Automated runs with no screen (our tests set PHYS367_HEADLESS=1) build the same widgets
    but don't run func, because live widget output can stall a headless run.
    """
    if os.environ.get("PHYS367_HEADLESS") == "1":
        return interact_manual(func, **controls)
    return interact(func, **controls)

BANDS = list("ugrizy")
BAND_COLORS = {"u": "tab:purple", "g": "tab:blue", "r": "tab:green",
               "i": "tab:orange", "z": "tab:red", "y": "tab:brown"}

## Part 1 · Depth from first principles

The 5σ limiting magnitude $m_5$ is the brightness of a point source that a single visit detects
at signal-to-noise 5. It depends on how many photons from the source reach the detector, how
many sky photons land in the same patch of pixels, and how much noise the camera adds. You will
build it in four steps: throughput, source photons, sky photons, and signal-to-noise.

### 1. The throughput chain

The fraction of photons at wavelength $\lambda$ that survive to become photoelectrons is a
product of components:

$$S_b(\lambda) = \underbrace{M_1 M_2 M_3}_{\text{mirrors}}\;\underbrace{L_1 L_2 L_3}_{\text{lenses}}
\;\underbrace{\text{QE}}_{\text{detector}}\;\underbrace{F_b}_{\text{filter}},
\qquad T_b(\lambda, X) = S_b(\lambda)\,A(\lambda, X)$$

where $S_b$ is the *hardware* throughput in band $b$ and $A(\lambda, X)$ is the atmosphere's
transmission at airmass $X$.

*Source: the Rubin systems-engineering throughput model, `lsst-pst/syseng_throughputs` at commit
`00570b3d39` (release 1.9, with silver coatings on all three mirrors). We rebuilt its
component products on a 1 nm grid (`tools/make_throughputs_extract.py` in the course repo);
they match syseng's own `buildHardwareAndSystem` output to about 1 part in 10⁶. Each component
includes syseng's loss terms (contamination, condensation), and the detector is the minimum of
the two CCD vendors' QE curves.* The atmosphere is tabulated at X = 1.0
(`atmos_10_aerosol.dat`, syseng's standard X = 1.0 atmosphere with an aerosol component) and
X = 1.2 (`pachonModtranAtm_12_aerosol.dat`), both from syseng. For other airmasses we
interpolate (and beyond X = 1.2, extrapolate) $\ln A$ linearly in airmass between the two curves. That interpolation is our choice, not syseng's.

**Where does this break?** Every factor here is a single curve for the whole focal plane and
the whole survey. Which of them would you expect to change across the field of view, from night
to night, or over ten years?

In [ ]:
SYSENG_SHA = "00570b3d391b5a8671d55341ed51b5e534dab6b4"   # syseng_throughputs, release 1.9

thr = load("throughputs")
dark_sed = load("darksky")
m5_ref = load("m5_reference").set_index("band")
print(header("throughputs"))

WAVE = thr["wavelength_nm"].to_numpy(dtype=float)   # nm, 1 nm grid, 300-1100 nm
DLAM = 1.0                                           # nm

def hardware(band):
    """Hardware throughput S_b (mirrors x lenses x detector x filter), no atmosphere."""
    return (thr["mirrors"] * thr["lenses"] * thr["detector"] * thr[f"filter_{band}"]).to_numpy()

def atmosphere(airmass=1.0):
    """Atmospheric transmission at this airmass.

    Exact at X = 1.0 and 1.2 (the two tabulated curves); elsewhere ln(A) is interpolated
    (or extrapolated) linearly in X between them.
    """
    if airmass < 1.0:
        raise ValueError("airmass must be >= 1 (1 = zenith)")
    a10 = thr["atmos_X1.0"].to_numpy()
    a12 = thr["atmos_X1.2"].to_numpy()
    ratio = np.divide(a12, a10, out=np.zeros_like(a10), where=a10 > 0)
    return np.clip(a10 * ratio ** ((airmass - 1.0) / 0.2), 0.0, 1.0)

def system(band, airmass=1.0):
    """Total throughput T_b = hardware x atmosphere, on the WAVE grid."""
    return hardware(band) * atmosphere(airmass)

In [ ]:
def show_components(airmass=1.0):
    """Plot every component of the throughput chain, then the total system curve per band."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
    keep = WAVE >= 320        # below ~320 nm nothing gets through (and the lens file has an edge artifact)
    w = WAVE[keep]
    for col, label, style in [("mirrors", "mirrors M1·M2·M3", "-"),
                              ("lenses", "lenses L1·L2·L3", "--"),
                              ("detector", "detector QE", "-.")]:
        ax1.plot(w, thr[col].to_numpy()[keep], "k", ls=style, lw=1.2, label=label)
    ax1.plot(w, atmosphere(airmass)[keep], color="0.6", lw=1.5, label=f"atmosphere, X = {airmass:g}")
    for b in BANDS:
        ax1.plot(w, thr[f"filter_{b}"].to_numpy()[keep], color=BAND_COLORS[b], lw=1, alpha=0.7)
    ax1.set_ylabel("throughput")
    ax1.set_ylim(0, 1.05)
    ax1.legend(loc="lower right", fontsize=8)
    ax1.set_title("Components (colored: the six filters)")
    for b in BANDS:
        ax2.plot(w, system(b, airmass)[keep], color=BAND_COLORS[b], label=b)
        ax2.plot(w, hardware(b)[keep], color=BAND_COLORS[b], lw=0.7, ls=":")
    ax2.set_ylabel("total throughput")
    ax2.set_xlabel("wavelength (nm)")
    ax2.set_title(f"System = product of all of the above (dotted: hardware only), X = {airmass:g}")
    ax2.legend(ncol=6, fontsize=8, loc="upper right")
    ax2.set_ylim(0, 0.7)
    plt.tight_layout()
    plt.show()

show_components()

**Read the plot.** Which component limits the u band? Which one sets the red edge of y? Where
does the atmosphere matter most?

### 2. Photons from a source: the zeropoint

An AB magnitude is defined against a flat spectrum, $f_\nu = 3631$ Jy at $m = 0$. The rate of
detected photons (photoelectrons) from such a source is

$$N_0 = A_\text{eff}\int \frac{f_\nu\,T_b(\lambda)}{h\lambda}\,d\lambda ,
\qquad N(m) = N_0\,10^{-0.4\,m}\ \ \text{per second}$$

where $A_\text{eff}$ is the effective collecting area. Divide $f_\nu$ by the photon energy
$h\nu$ to count photons, and use $d\nu/\nu = d\lambda/\lambda$ to integrate in wavelength.
$2.5\log_{10}N_0$ is the magnitude that gives one photoelectron per second.

*Source: AB system (Oke & Gunn 1983); the same sum as rubin_sim's `Sed.calc_adu`, which
syseng uses. The collecting area is rubin_sim's default, $A_\text{eff} = \pi(6.423\ \text{m}/2)^2$
= 32.4 m². Rubin's primary mirror is 8.4 m across; the central obscuration makes the effective aperture
smaller (Ivezić et al. 2019 quote 6.5 m effective; rubin_sim's slightly smaller value is used
throughout).*

**Where does this break?** Real stars and galaxies don't have flat $f_\nu$. What happens to
the photon count for a very red or very blue source with the same AB magnitude?

In [ ]:
H_ERG_S = 6.626068e-27           # Planck constant, erg s (rubin_sim's value)
C_NM_S = 2.99792458e17           # speed of light, nm/s
AB_ZERO_FNU = 3631e-23           # erg s^-1 cm^-2 Hz^-1 (3631 Jy)
RUBIN_AREA_M2 = np.pi * (6.423 / 2) ** 2   # rubin_sim PhotometricParameters default

def zeropoint(band, airmass=1.0, area_m2=None, throughput=None):
    """Photoelectrons per second from an m = 0 AB source (flat f_nu = 3631 Jy).

    Uses the full system throughput (hardware x atmosphere) unless you pass `throughput`.
    """
    area_cm2 = (RUBIN_AREA_M2 if area_m2 is None else area_m2) * 1e4
    T = system(band, airmass) if throughput is None else throughput
    return area_cm2 * np.sum(AB_ZERO_FNU * T / (H_ERG_S * WAVE)) * DLAM

print("Zeropoint: magnitude giving 1 photoelectron/s (X = 1.0)")
print(f"{'band':>4} {'this notebook':>14} {'syseng makeM5':>14}")
for b in BANDS:
    print(f"{b:>4} {2.5 * np.log10(zeropoint(b)):14.3f} {m5_ref.loc[b, 'Zp_t']:14.3f}")

### 3. Photons from the sky

The night sky is not dark. Airglow, zodiacal light and scattered light give a surface
brightness $F_\lambda^\text{sky}$ per arcsec². The rate of sky photoelectrons per arcsec² is

$$B = A_\text{eff}\int F_\lambda^\text{sky}\,\frac{\lambda}{hc}\,S_b(\lambda)\,d\lambda$$

through the **hardware only**: the sky glow is made in the atmosphere, and syseng's dark-sky
spectrum already describes the light arriving at the telescope.

*Source: `siteProperties/darksky.dat` in syseng (same commit), a dark, moonless sky at zenith.
The file's header gives units of erg s⁻¹ cm⁻² nm⁻¹ only. We treat it as per arcsec² because
that is how syseng's `makeM5` uses it (it multiplies by the pixel area in arcsec²).* To try a
brighter sky, `sky_counts(band, sky_mag=...)` rescales the same spectrum to the surface brightness
you give it, in mag/arcsec².

**Where does this break?** This is one dark sky at zenith. What does the Moon do, and would it
matter equally in every band? What about twilight, or looking toward the Galactic plane or the
ecliptic?

In [ ]:
def sky_counts(band, sky_mag=None, area_m2=None):
    """Sky photoelectrons per second per arcsec^2, through the hardware (no atmosphere).

    sky_mag=None uses the dark-sky spectrum as is; otherwise the same spectrum is rescaled
    to that surface brightness (AB mag/arcsec^2 through the hardware).
    """
    area_cm2 = (RUBIN_AREA_M2 if area_m2 is None else area_m2) * 1e4
    flam = dark_sed["flambda_erg_s_cm2_nm"].to_numpy()
    dark = area_cm2 * np.sum(flam * WAVE / (H_ERG_S * C_NM_S) * hardware(band)) * DLAM
    if sky_mag is None:
        return dark
    return dark * 10 ** (-0.4 * (sky_mag - sky_mag_dark(band)))

def sky_mag_dark(band):
    """Surface brightness of the dark-sky spectrum in this band (AB mag/arcsec^2)."""
    return -2.5 * np.log10(sky_counts(band) / zeropoint(band, throughput=hardware(band)))

print("Dark-sky surface brightness at zenith (mag/arcsec^2)")
print(f"{'band':>4} {'this notebook':>14} {'syseng makeM5':>14} {'sky e-/s/pixel':>15}")
for b in BANDS:
    print(f"{b:>4} {sky_mag_dark(b):14.3f} {m5_ref.loc[b, 'skyMag']:14.3f} "
          f"{sky_counts(b) * 0.2**2:15.1f}")

### 4. Signal-to-noise and $m_5$

A visit is $n_\text{exp}$ exposures of $t_\text{exp}$ seconds each, so the total open-shutter time is
$t = n_\text{exp}t_\text{exp}$. For a point source giving $C$ photoelectrons, spread over a PSF
that covers $n_\text{eff}$ pixels,

$$\frac{S}{N} = \frac{C}{\sqrt{C + n_\text{eff}\,\big(B\,p^2\,t + n_\text{exp}\,\sigma_\text{read}^2 + D\,t\big)}},
\qquad n_\text{eff} = 2.266\left(\frac{\text{FWHM}_\text{eff}}{p}\right)^2$$

Here $p$ is the pixel scale, $\sigma_\text{read}$ the read noise per pixel per exposure, and
$D$ the dark current. Set $S/N = 5$ and solve the quadratic for $C$. Then
$m_5 = -2.5\log_{10}\!\big[C/(N_0\,t)\big]$.

For a Gaussian PSF of width $\sigma$, $n_\text{eff} = 4\pi\sigma^2/p^2 = 2.266\,(\text{FWHM}/p)^2$:
the number of pixels whose noise an optimal (PSF-weighted) measurement effectively averages over.

*Source: the LSST signal-to-noise document LSE-40 (eq. 27 for $n_\text{eff}$, eq. 45 for
$m_5$), as coded in rubin_sim's `signaltonoise.calc_m5` and used by syseng's `makeM5`.*

| parameter | value | source |
|---|---|---|
| effective area | π(6.423 m / 2)² = 32.4 m² | rubin_sim 2.6.2 `phot_utils/photometric_parameters.py`, `DefaultPhotometricParameters` |
| pixel scale $p$ | 0.2″ | same file |
| read noise | 8.8 e⁻ per pixel per exposure | same file (from the camera spec, LSE-30) |
| dark current | 0.2 e⁻/s per pixel | same file |
| gain | 1 (we count electrons) | syseng `makeM5` sets gain = 1; rubin_sim's default is 2.3 e⁻/ADU, which cancels when you work in electrons |
| FWHM$_\text{eff}$ | u 0.92, g 0.87, r 0.83, i 0.80, z 0.78, y 0.76″ at zenith, × $X^{0.6}$ | syseng `m5Utils.fwhm_eff_zenith` (fiducial seeing from the LSST overview paper, Ivezić et al. 2019) |
| visit | 2 × 15 s in every band | the reference convention used here; syseng's `makeM5` default is 1 × 30 s in u |

The reference table `m5_reference.csv` was made by running syseng's own `makeM5` at the pinned
commit with exactly these settings at X = 1.0 (see its header). Your function should agree with it.

**Where does this break?** The formula assumes a faint point source on a smooth, perfectly
subtracted sky, with a Gaussian-like PSF. Name a kind of source, or a part of the sky, where
each of those assumptions fails. What would happen to $m_5$?

In [ ]:
PIXEL_SCALE = 0.2                       # arcsec per pixel (rubin_sim default)
FWHM_EFF_ZENITH = {"u": 0.92, "g": 0.87, "r": 0.83, "i": 0.80, "z": 0.78, "y": 0.76}  # arcsec, syseng

def m5(band, t_exp=15, n_exp=2, fwhm_eff=None, sky_mag=None, read_noise=8.8, dark=0.2,
       area_m2=None, airmass=1.0):
    """5-sigma point-source depth (AB mag) of one visit of n_exp exposures of t_exp seconds.

    fwhm_eff=None uses syseng's fiducial zenith FWHM_eff x airmass^0.6 (arcsec).
    sky_mag=None uses the dark-sky spectrum; otherwise the sky in mag/arcsec^2.
    read_noise in e-/pixel/exposure; dark in e-/pixel/s; area_m2=None means Rubin's 32.4 m^2.
    """
    if fwhm_eff is None:
        fwhm_eff = FWHM_EFF_ZENITH[band] * airmass ** 0.6
    t = t_exp * n_exp                                        # open-shutter seconds
    n_eff = 2.266 * (fwhm_eff / PIXEL_SCALE) ** 2            # pixels in the PSF (LSE-40 eq. 27)
    sky_per_pixel = sky_counts(band, sky_mag, area_m2) * PIXEL_SCALE**2 * t
    var_bkg = n_eff * (sky_per_pixel + n_exp * read_noise**2 + dark * t)
    snr = 5.0
    counts_5sig = snr**2 / 2 + np.sqrt(snr**4 / 4 + snr**2 * var_bkg)   # solves S/N = 5 for C
    return -2.5 * np.log10(counts_5sig / (zeropoint(band, airmass, area_m2) * t))

print("Single-visit m5, 2 x 15 s, X = 1.0, dark sky, fiducial seeing")
print(f"{'band':>4} {'this notebook':>14} {'syseng makeM5':>14} {'difference':>11}")
for b in BANDS:
    mine, ref = m5(b), m5_ref.loc[b, "m5"]
    print(f"{b:>4} {mine:14.3f} {ref:14.3f} {mine - ref:+11.4f}")

### Design → as-built → as-scheduled

Three numbers for the same thing, the depth of one r-band visit:

| r-band $m_5$ | what it is | source |
|---|---|---|
| **24.7** | the design ("fiducial") depth | Ivezić et al. 2019, Table 1 (SRD design specification, fiducial zenith); Monday's worksheet |
| **≈ 24.48** | the as-built hardware (release 1.9) at the reference settings: 2 × 15 s, zenith, dark sky, FWHM$_\text{eff}$ 0.83″ | this notebook's `m5("r")`; syseng `makeM5` gives 24.479 |
| **24.06** | the median r-band visit in the simulated ten-year survey | baseline v5.3.3 simulation (Notebook A); median FWHM$_\text{eff}$ 1.03″ (`seeingFwhmEff`: use it in `m5()` as FWHM$_\text{eff}$, don't convert), airmass 1.18, sky 21.0 mag/arcsec² |

In [ ]:
M5_DESIGN_R = 24.7        # Ivezic et al. 2019, Table 1 (SRD design specification, fiducial zenith; also the worksheet)
M5_SIM_MEDIAN_R = 24.06   # median r-band visit, baseline v5.3.3 simulation (Notebook A)

print(f"design (Ivezic+2019 Table 1, SRD spec):    {M5_DESIGN_R:.2f}")
print(f"as-built hardware (this notebook, m5('r')): {m5('r'):.2f}")
print(f"median simulated visit (baseline v5.3.3): {M5_SIM_MEDIAN_R:.2f}")

**Account for each step.** From 24.7 to ≈ 24.48: what went into 24.7, and which of those inputs
differ from the as-built calculation? From ≈ 24.48 to 24.06: call `m5("r", ...)`
with the simulation's median seeing, airmass and sky. How much of the drop does each one explain
on its own? Is the depth at the median conditions the same thing as the median depth?

**Your science.** Your plan in Notebook A used 24.7. Which of your numbers change if you use
24.06 instead, and by how much?

### Read noise, and why the u band is different

The reference visit is two 15 s exposures. syseng's own `makeM5` defaults to a single 30 s
exposure in u, and two 15 s exposures in the other bands. The cell below computes, with your
`m5()`, how much deeper one 30 s exposure is than two 15 s exposures in each band. Next to it:
the sky electrons per pixel in one 15 s exposure, and syseng's `dCm_double` from the reference table: the depth that a visit of twice the
exposure time still loses to camera noise, compared with a noiseless camera.

In [ ]:
RN = 8.8   # e- per pixel per exposure
print(f"{'band':>4} {'sky e-/pix/15s':>15} {'read noise^2':>13} "
      f"{'m5(1x30s) - m5(2x15s)':>22} {'dCm_double (syseng)':>20}")
for b in BANDS:
    sky_pix = sky_counts(b) * PIXEL_SCALE**2 * 15
    gain_30 = m5(b, t_exp=30, n_exp=1) - m5(b, t_exp=15, n_exp=2)
    print(f"{b:>4} {sky_pix:15.1f} {RN**2:13.1f} {gain_30:22.3f} "
          f"{m5_ref.loc[b, 'dCm_double']:20.3f}")

**Why u?** Why does the u band gain the most from one 30 s exposure, and lose the most to read
noise? Compare the first two columns. What would you change about the camera, or the visit, to
fix it, and what would each fix cost?

**Where does this break?** Two exposures per visit let you reject cosmic rays and catch
fast-moving objects. What do you give up with one 30 s exposure?

### Your camera, with sliders

Change the telescope and camera. The aperture is the *effective* diameter (Rubin: 6.423 m).
"Sky brighter by" makes the sky brighter than the dark sky by that many mag/arcsec². The seeing
slider is an *offset* from the band's fiducial FWHM$_\text{eff}$ (0.92″ in u down to 0.76″ in
y), so 0 reproduces the reference; +0.2 means 0.2″ worse than fiducial. The plot
shows $m_5$ against open-shutter time for your camera and for the Rubin reference.

If the sliders don't appear (e.g. in a PDF), call
`show_m5(band, aperture_m=..., read_noise=..., fwhm_eff=..., sky_mag=..., t_exp=..., n_exp=...)`
directly; that's also what to use for your hand-in figure.

In [ ]:
def show_m5(band="r", aperture_m=6.423, read_noise=8.8, fwhm_eff=None, sky_mag=None,
            t_exp=15, n_exp=2, airmass=1.0):
    """Print m5 for one visit with these settings and plot m5 vs exposure time. Returns m5."""
    area = np.pi * (aperture_m / 2) ** 2
    mine = m5(band, t_exp, n_exp, fwhm_eff, sky_mag, read_noise, 0.2, area, airmass)
    ref = m5(band)
    fwhm_used = FWHM_EFF_ZENITH[band] * airmass ** 0.6 if fwhm_eff is None else fwhm_eff
    sky_used = sky_mag_dark(band) if sky_mag is None else sky_mag
    print(f"{band} band: {n_exp} x {t_exp:g} s, aperture {aperture_m:.2f} m, read noise "
          f"{read_noise:.1f} e-, FWHM_eff {fwhm_used:.2f}\", sky {sky_used:.2f} mag/arcsec^2, "
          f"X = {airmass:.2f}")
    print(f"  m5 = {mine:.2f}   (Rubin reference, 2 x 15 s: {ref:.2f}; "
          f"difference {mine - ref:+.2f} mag)")

    t_grid = np.geomspace(1, 300, 60)                   # exposure time per exposure, s
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(t_grid * n_exp, [m5(band, t, n_exp, fwhm_eff, sky_mag, read_noise, 0.2, area, airmass)
                             for t in t_grid], color=BAND_COLORS[band], label="your camera")
    ax.plot(t_grid * 2, [m5(band, t, 2) for t in t_grid], "k--", lw=1,
            label="Rubin reference (2 exposures)")
    ax.scatter([t_exp * n_exp], [mine], color=BAND_COLORS[band], zorder=3)
    ax.scatter([30], [ref], color="k", zorder=3)
    ax.set_xscale("log")
    ax.set_xlabel("open-shutter time per visit (s)")
    ax.set_ylabel(f"{band}-band $m_5$ (AB mag)")
    ax.legend(fontsize=8)
    plt.show()
    return mine

In [ ]:
from ipywidgets import Dropdown, FloatSlider, IntSlider

_W = dict(continuous_update=False, style={"description_width": "initial"})

def _m5_widget(band, aperture_m, read_noise, seeing_offset, sky_brighter, t_exp, n_exp):
    fwhm_eff = FWHM_EFF_ZENITH[band] + seeing_offset     # offset from the band's fiducial
    show_m5(band, aperture_m, read_noise, fwhm_eff, sky_mag_dark(band) - sky_brighter,
            t_exp, n_exp)                               # don't echo the returned m5

sliders(_m5_widget,
        band=Dropdown(options=BANDS, value="r", description="band"),
        aperture_m=FloatSlider(value=6.423, min=0.5, max=12.0, step=0.1,
                               description="aperture (m)", **_W),
        read_noise=FloatSlider(value=8.8, min=0.0, max=30.0, step=0.2,
                               description="read noise (e⁻)", **_W),
        seeing_offset=FloatSlider(value=0.0, min=-0.4, max=1.6, step=0.01,
                                  description="seeing relative to fiducial (″)", **_W),
        sky_brighter=FloatSlider(value=0.0, min=0.0, max=5.0, step=0.1,
                                 description="sky brighter by (mag)", **_W),
        t_exp=IntSlider(value=15, min=1, max=300, step=1,
                        description="t_exp (s)", **_W),
        n_exp=IntSlider(value=2, min=1, max=10, step=1,
                        description="n_exp", **_W));

**Where are you limited?** For each band, find the exposure time at which read noise stops
mattering. Does it depend on the sky brightness? On the seeing?

**Buy one thing.** You can afford one upgrade: 10% more aperture, half the read noise, or
0.1″ better seeing. Which buys the most depth in r? In u? Is it the same answer?

### Étendue: aperture × field of view

Depth per visit depends on the aperture. How much sky you can cover depends on the field of view.
A survey telescope is rated by their product, the **étendue** $A_\text{eff}\,\Omega$.
Here is a toy argument for why.

1. If every visit is limited by sky noise, $m_5$ depends on $A_\text{eff}\,t$. So reaching the
   same depth takes $t \propto 1/A_\text{eff}$.
2. Covering the same area takes $\propto 1/\Omega$ pointings.
3. So the worksheet's ten-year budget, Area × visits × $(t/30\,\text{s}) \approx 1.5\times10^7$
   (Monday's worksheet, Notebook A), scales as $A_\text{eff}\,\Omega$ when every visit has
   Rubin's 30 s depth.

*Sources: Rubin's field of view is 9.6 deg² (Ivezić et al. 2019); the effective area is rubin_sim's
default, as above. This is a toy model.* Our Rubin étendue, ≈ 311 m² deg², is rubin_sim's
effective area (6.423 m effective diameter) × 9.6 deg²; Ivezić et al. 2019 Table 1 quotes
319 m² deg² because it uses a 6.5 m effective diameter.

**Where does this break?** The argument assumes every visit is sky-noise limited, the overhead
per visit is negligible, and the telescope has the same number of good nights. Which of these
fails first for a much smaller telescope? For a much smaller field?

In [ ]:
BUDGET = 1.5e7               # area x visits x (t/30 s) over 10 yr, Monday's worksheet
RUBIN_DIAM_EFF_M = 6.423     # effective aperture, m (rubin_sim default area)
RUBIN_FOV_DEG2 = 9.6         # field of view, deg^2 (Ivezic et al. 2019)

def survey_budget(diam_eff_m, fov_deg2):
    """Toy: the worksheet's 10-yr budget, scaled by etendue relative to Rubin."""
    etendue = np.pi * (diam_eff_m / 2) ** 2 * fov_deg2
    etendue_rubin = np.pi * (RUBIN_DIAM_EFF_M / 2) ** 2 * RUBIN_FOV_DEG2
    return BUDGET * etendue / etendue_rubin

print("Toy: ten-year budget at Rubin's 30 s single-visit depth, scaled by etendue")
print(f"{'telescope':<28} {'etendue (m^2 deg^2)':>20} {'budget':>9} "
      f"{'visits/field @18k deg^2':>24} {'r m5, 2x15 s':>13}")
for label, d, fov in [("Rubin (6.423 m, 9.6 deg^2)", RUBIN_DIAM_EFF_M, RUBIN_FOV_DEG2),
                      ("4 m aperture, 9.6 deg^2", 4.0, RUBIN_FOV_DEG2),
                      ("6.423 m, 1 deg^2 field", RUBIN_DIAM_EFF_M, 1.0)]:
    et = np.pi * (d / 2) ** 2 * fov
    b = survey_budget(d, fov)
    print(f"{label:<28} {et:20.0f} {b:9.2e} {b / 18000:24.0f} "
          f"{m5('r', area_m2=np.pi * (d / 2) ** 2):13.2f}")

**What would it do to the survey?** Take your team's plan from Notebook A. With a 4 m telescope,
or with a 1 deg² field, would you rather keep the area and lose visits, or keep the visits and
lose area? Which science case suffers most from each?

> **Go deeper.** The atmosphere here interpolates between two tabulated curves. The textbook
> alternative is Beer–Lambert: $A(\lambda, X) = A(\lambda, 1)^X$. Compute
> `thr["atmos_X1.0"]**1.2` and compare it with `thr["atmos_X1.2"]`. Where do they differ, and why
> might a single exponent not describe every part of the atmosphere?

## Part 2 · Seeing → weak lensing

Weak lensing measures a tiny, coherent stretch in the shapes of distant galaxies. The telescope
and the atmosphere blur every galaxy with the point-spread function (PSF). A galaxy much smaller
than the PSF comes out looking like the PSF, and its shape tells you almost nothing. So the
seeing decides how many galaxies you can use.

Here are the image-quality numbers you'll compare. FWHM is the full width at half maximum of the PSF.

| PSF FWHM | what it is | source |
|---|---|---|
| **0.7″** | the design seeing, rounded (Monday's worksheet) | Ivezić et al. 2019, Table 2: expected delivered median zenith seeing (FWHM) 0.71″ in i (FWHM$_\text{eff}$ 0.80″) |
| **0.73″** | first-night median seeing (atmosphere only: what you'd get if the telescope added nothing) | Phil Marshall, Sep 23 overview |
| **0.91″** | first-night median delivered image quality | Phil Marshall, Sep 23 overview |
| **1.1″** | median delivered image quality, first ~15 days in survey | Phil Marshall, Sep 23 overview |

On the first night Marshall also reported a median PSF ellipticity of 0.078 (the requirement
is 0.04). For the first ~15 days he listed, as separate items: total optical-system
contribution 0.45″; dome seeing 0.3–0.4″; tracking error ~0.1″. These are early commissioning
numbers.

### 1. A toy model for usable galaxies

You need three ingredients.

**How many galaxies.** The counts from Notebook A, $N(<i) = 46\times10^{0.31(i-25)}$ arcmin⁻²
(*LSST Science Book 2009, eq. 3.7, a fit to CFHTLS Deep counts over 20.5 < i < 25.5*). Down to
the Science Book's "gold sample" limit, $i < 25.3$ (S/N > 20 for point sources in median
conditions), that is ≈ 57 arcmin⁻² from eq. 3.7 (the Science Book quotes ≈ 55, §3.7.2).

**How big they are.** Miller et al. 2013 (the CFHTLenS shape paper, Appendix B, §B1, eq. B1)
fit the median disk scale length $r_d$ of galaxies measured with Hubble (Simard et al. 2002)
against magnitude, over $18.5 < i_{814} < 25.5$:

$$\ln\!\left(\frac{r_d}{\text{arcsec}}\right) = -1.145 - 0.269\,(i - 23)$$

Around that median, the sizes scatter as $p(r) \propto r\,\exp[-(r/a)^{4/3}]$ (same appendix).
Fainter galaxies are smaller: the median $r_d$ is ≈ 0.32″ at $i = 23$ and ≈ 0.17″ at $i = 25.3$.

**Which ones are resolved.** Treat the galaxy and the PSF as Gaussians of width
$\sigma_\text{gal}$ and $\sigma_\text{PSF}$. The *resolution factor* is

$$R_2 = \frac{\sigma_\text{gal}^2}{\sigma_\text{gal}^2 + \sigma_\text{PSF}^2}$$

It is 1 for a galaxy much larger than the PSF and 0 for one much smaller. A galaxy is
*usable* if $R_2 > 1/3$. *Source: the cut used for the SDSS lensing shape catalog,
Mandelbaum et al. 2005, §2, eq. 10; the Gaussian form of $R_2$ is from the caption of their
Fig. 5.*

**Our choices (not from any paper).** An exponential disk's half-light radius is
$1.678\,r_d$. We replace each galaxy by a Gaussian with the same half-light radius
($\sigma_\text{gal} = 1.678\,r_d/1.1774$). We use the PSF FWHM in the table above as a Gaussian
FWHM ($\sigma_\text{PSF} = \text{FWHM}/2.355$). We take Hubble's $i_{814}$ to be LSST's
$i$. We ignore bulges, ellipticity, blending, and the S/N of the shape measurement.
The Gaussian matching matters: matching second moments instead ($\sigma_\text{gal} =
\sqrt{3}\,r_d$, the rms radius of an exponential disk) gives ≈ 45 arcmin⁻² at 0.7″ rather
than ≈ 41 (this notebook's functions: `usable_density(0.7, sigma_per_rd=np.sqrt(3))`).

**Where does this break?** The size relation comes from disk-dominated galaxies in one small
Hubble field, measured along the major axis. Mandelbaum et al. warn that the Gaussian form of
$R_2$ fails for bulge-dominated (de Vaucouleurs) profiles. Which way would each of these push
the number of usable galaxies?

In [ ]:
FWHM_DESIGN = 0.7        # arcsec: design seeing, rounded (Ivezic et al. 2019 Table 2: 0.71" in i; worksheet)
SEEING_MARKERS = {       # PSF FWHM in arcsec
    "design": 0.7,                           # Ivezic et al. 2019 Table 2 (0.71" in i), rounded
    "first-night seeing (atmosphere only)": 0.73,   # Marshall, Sep 23: if the telescope added nothing
    "first night delivered": 0.91,           # Marshall, Sep 23: median delivered image quality
    "first ~15 days delivered": 1.1,         # Marshall, Sep 23: median delivered image quality
}
I_GOLD = 25.3            # Science Book 3.7.2 "gold sample" limit, i < 25.3
I_BRIGHT = 18.5          # bright end of the Miller et al. size fit; brighter galaxies are few (< 0.5 arcmin^-2)
N_EFF_CHANG = 37.0       # Chang et al. 2013: LSST n_eff (r+i, 10 yr), before blending and masking

from math import erfc
_erfc = np.vectorize(erfc)

def counts_per_mag(i):
    """dN/di in galaxies per arcmin^2 per mag: derivative of Science Book eq. 3.7."""
    return np.log(10) * 0.31 * 46 * 10 ** (0.31 * (i - 25))

def median_scalelength(i):
    """Median disk scale length r_d (arcsec) at i-band magnitude i: Miller et al. 2013, eq. B1."""
    return np.exp(-1.145 - 0.269 * (i - 23))

def frac_resolved(i, fwhm, r_min=1/3, median_over_a=1.134, sigma_per_rd=1.678 / 1.1774):
    """Fraction of galaxies at magnitude i with resolution factor R2 > r_min, for a PSF of this FWHM.

    Galaxies: Miller et al. 2013 sizes, each replaced by a Gaussian of width
    sigma_per_rd * r_d (default: the same half-light radius). PSF: a Gaussian with this FWHM
    (arcsec). median_over_a sets the size scale a = median r_d / median_over_a.
    """
    sigma_psf = fwhm / 2.3548
    sigma_gal_min = sigma_psf * np.sqrt(r_min / (1 - r_min))   # R2 > r_min  <=>  sigma_gal > this
    rd_min = sigma_gal_min / sigma_per_rd                     # the scale length with that sigma
    # Miller et al.: p(r) ~ r exp[-(r/a)^(4/3)], with a set so the median is eq. B1.
    # For this distribution the median is 1.134 a.
    a = median_scalelength(i) / median_over_a
    x = (rd_min / a) ** (4 / 3)
    # P(r > rd_min): with x = (r/a)^(4/3) this is a gamma distribution of shape 3/2,
    # whose upper tail has the closed form below.
    return _erfc(np.sqrt(x)) + 2 * np.sqrt(x / np.pi) * np.exp(-x)

for i in [21, 23, 24, 25, 25.3]:
    print(f"i = {i:4}: median r_d = {median_scalelength(i):.2f}\"   "
          f"resolved at 0.7\": {frac_resolved(i, 0.7):.2f}   at 1.1\": {frac_resolved(i, 1.1):.2f}")

### 2. Worse seeing also costs depth

From Part 1: a wider PSF spreads a point source over more sky pixels, so $m_5$ gets brighter.
The next cell uses your `m5()` to shift the magnitude limit. At the design seeing it is
$i < 25.3$, and it moves by exactly as much as the single-visit $m_5$ in i moves. That
pretends *every* visit had this seeing.

One conversion is needed. `m5()` takes FWHM$_\text{eff}$ (the width of the equivalent
single Gaussian, Part 1), while the numbers in the table are measured FWHMs. rubin_sim
converts them with FWHM = 0.822 FWHM$_\text{eff}$ + 0.052″ (*rubin_sim 2.6.2,
`phot_utils/signaltonoise.py`, `fwhm_geom2_fwhm_eff`*).

**Where does this break?** The $m_5$ loss is for point sources. A resolved galaxy is already
spread over its own area. Does seeing cost you more or less depth for a large galaxy than for a
star? And a real survey mixes good and bad nights in one coadd.

In [ ]:
def fwhm_to_eff(fwhm):
    """Measured PSF FWHM -> FWHM_eff for m5() (rubin_sim fwhm_geom2_fwhm_eff), arcsec."""
    return (fwhm - 0.052) / 0.822

def i_limit(fwhm):
    """Toy magnitude limit: i < 25.3 at the design seeing, shifted by the change in i-band m5."""
    return I_GOLD + m5("i", fwhm_eff=fwhm_to_eff(fwhm)) - m5("i", fwhm_eff=fwhm_to_eff(FWHM_DESIGN))

def usable_density(fwhm, r_min=1/3, depth=True, **size_kw):
    """Toy: galaxies per arcmin^2 with R2 > r_min, down to the magnitude limit.

    depth=True moves the limit with the seeing (i_limit); depth=False keeps i < 25.3.
    This is a count of resolved galaxies, not Chang et al.'s weighted n_eff.
    size_kw (median_over_a, sigma_per_rd) are passed to frac_resolved.
    """
    i_max = i_limit(fwhm) if depth else I_GOLD
    edges = np.linspace(I_BRIGHT, i_max, 401)
    mid = 0.5 * (edges[1:] + edges[:-1])
    return float(np.sum(counts_per_mag(mid) * frac_resolved(mid, fwhm, r_min, **size_kw) * np.diff(edges)))

n_design = usable_density(FWHM_DESIGN)
print(f"{'':38} {'FWHM':>6} {'i limit':>8} {'resolved only':>14} {'+ depth loss':>13} {'vs design':>10}")
for name, f in SEEING_MARKERS.items():
    n_d = usable_density(f)
    print(f"{name:38} {f:5.2f}\" {i_limit(f):8.2f} {usable_density(f, depth=False):14.1f} "
          f"{n_d:13.1f} {n_d / n_design:10.2f}")
print("(galaxies per arcmin^2; toy model)")
print(f"Chang et al. 2013 n_eff: {N_EFF_CHANG:.0f} arcmin^-2. Science Book 3.7.2: ~40 arcmin^-2 (+/- 20%).")

### 3. Usable galaxies vs. seeing

The left panel shows the usable density against PSF FWHM, with and without the depth loss.
The markers are the four seeing values from the table. The gray line is the published LSST
estimate: Chang et al. 2013 (MNRAS 434, 2121) found $n_\text{eff} \approx 37$ arcmin⁻² for
r+i over ten years before blending and masking, 31 after rejecting serious blends, and 26
after a further 15% loss to masking. Theirs is a full simulation with a weighted $n_\text{eff}$;
ours is a toy count, so compare shapes and ratios, not the last digit. The right panel shows
which magnitudes you lose. Left of the design seeing, the depth-shifted limit passes
$i = 25.5$, the faint end of the eq. 3.7 fit, so that part of the solid curve is an
extrapolation.

If the slider doesn't appear, call `show_lensing(fwhm, r_min=1/3, depth=True)` directly.

In [ ]:
def show_lensing(fwhm=0.91, r_min=1/3, depth=True):
    """Plot usable density vs PSF FWHM, and what is lost by magnitude at this FWHM. Returns the density."""
    grid = np.linspace(0.4, 1.6, 61)
    n_here = usable_density(fwhm, r_min, depth)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

    ax1.plot(grid, [usable_density(f, r_min, True) for f in grid], "k", label="resolved + depth loss")
    ax1.plot(grid, [usable_density(f, r_min, False) for f in grid], "k--", lw=1,
             label="resolved only (i < 25.3)")
    ax1.axhline(N_EFF_CHANG, color="0.6", lw=1, label="Chang et al. 2013 $n_{eff}$ = 37")
    for (name, f), color in zip(SEEING_MARKERS.items(), ["tab:blue", "tab:green", "tab:orange", "tab:red"]):
        ax1.axvline(f, color=color, ls=":", lw=1)
        ax1.plot(f, usable_density(f, r_min, depth), "o", color=color, label=f"{name} ({f}\")")
    ax1.plot(fwhm, n_here, "k*", ms=12, label=f"you: {fwhm:.2f}\" -> {n_here:.1f}")
    ax1.set_xlabel("PSF FWHM (arcsec)")
    ax1.set_ylabel("usable galaxies per arcmin$^2$ (toy)")
    ax1.set_ylim(0, None)
    ax1.legend(fontsize=7)

    i_max = i_limit(fwhm) if depth else I_GOLD
    mags = np.linspace(I_BRIGHT, 26.0, 200)
    ax2.plot(mags, counts_per_mag(mags), "0.5", label="all galaxies (eq. 3.7)")
    ok = mags <= i_max
    ax2.fill_between(mags[ok], 0, (counts_per_mag(mags) * frac_resolved(mags, fwhm, r_min))[ok],
                     color="tab:blue", alpha=0.5, label=f"usable at {fwhm:.2f}\"")
    ax2.axvline(i_max, color="k", ls="--", lw=1, label=f"i limit {i_max:.2f}")
    ax2.set_xlabel("i magnitude")
    ax2.set_ylabel("galaxies per arcmin$^2$ per mag")
    ax2.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    return n_here

_ = show_lensing(1.1)

In [ ]:
from ipywidgets import Checkbox

def _lensing_widget(fwhm, r_min, depth):
    show_lensing(fwhm, r_min, depth)          # don't echo the returned density

sliders(_lensing_widget,
        fwhm=FloatSlider(value=0.91, min=0.4, max=1.6, step=0.01,
                         description="PSF FWHM (″)", **_W),
        r_min=FloatSlider(value=1/3, min=0.1, max=0.6, step=0.01,
                          description="resolution cut R₂ >", **_W),
        depth=Checkbox(value=True, description="include depth loss"));

**What does 1.1″ cost?** Compare the usable density at 1.1″ (first ~15 days, delivered) with
0.91″ (first night, delivered) and with the 0.7″ design. How much of the loss comes from
resolution and how much from depth? How does the answer change if you move the
resolution cut?

**How rough is "figure of merit ∝ $n_\text{eff}$"?** Very rough. The error on the lensing power
spectrum (Chang et al. 2013, eq. 11) is proportional to $P(\ell) + \sigma_\text{SN}^2/n_\text{eff}$ (plus
systematics), where $P$ is the signal and $\sigma_\text{SN} \approx 0.26$ is the scatter in
intrinsic galaxy shapes. $n_\text{eff}$ enters only through the second term. So treat
"FoM ∝ $n_\text{eff}$" as a way to rank options, not a number to quote. On which scales does
losing galaxies cost you almost nothing, and where does it cost more than proportionally?

**Where does the extra come from?** On the first night the atmosphere alone gave 0.73″ and
the delivered image quality was 0.91″. For the first ~15 days Marshall listed: delivered
1.1″; total optical-system contribution 0.45″; dome seeing 0.3–0.4″; tracking error ~0.1″.
Independent blurs add roughly in quadrature. Do dome seeing and tracking add up to 0.45″? What
would the first night's optical-system contribution have been? What median seeing would make
0.45″ consistent with 1.1″? Which contributions can be fixed, and which can't?

**Why does PSF ellipticity matter?** The first-night median PSF ellipticity was 0.078, against
a requirement of 0.04. The lensing shear you're after is a few percent at most (Chang et al.
2013 note it is over an order of magnitude smaller than the 0.26 shape scatter). A galaxy at the
cut, $R_2 = 1/3$, is two-thirds PSF by second moment. If your model of the PSF's ellipticity is
off by a small fraction, what happens to the shear you infer? Why would a small PSF ellipticity
be easier to correct than a large one?

> **Go deeper.** Miller et al. write the size distribution's scale as $a = r_d/0.833$. For
> $p(r) \propto r\,e^{-(r/a)^{4/3}}$, the median is $1.134\,a$, so $a = r_d/0.833$ would put the
> median at $1.36\,r_d$, not at $r_d$. The paper also says it chose $a$ to match the median of
> eq. B1, and we follow that. We can't tell from the paper which constant the lensfit code
> actually used. With our choice the design value is ≈ 41 arcmin⁻²; with the printed constant,
> `usable_density(0.7, median_over_a=0.833)`, it is ≈ 47 (this notebook's functions).
> Check the median of $r\,e^{-(r/a)^{4/3}}$ numerically. For which $\alpha$ is the median of
> $r\,e^{-(r/a)^\alpha}$ equal to $0.833\,a$? (Try $\alpha = 2$.) How does the spread between
> these choices compare with the gap between our toy and Chang et al.?

## Part 3 · The focal plane

So far every visit has been one number: one depth, one seeing. On the sky a visit is a
picture taken by 189 separate CCDs, and between the CCDs there is no data. This part draws the
focal plane, measures how much of it is silicon, and shows what the gaps do to a survey.

**The layout.** The LSST Camera has 189 science CCDs, each 4k × 4k with 10 µm pixels, grouped
into 21 "rafts" of 3 × 3 CCDs. Each raft has its own electronics. The four corners of the
5 × 5 grid hold special rafts with guide sensors (which track the stars during an exposure)
and wavefront sensors (split, half-size CCDs held just above and below focus, used to keep the
optics aligned). Together that is 3.2 gigapixels (3.2 counts the full 4k × 4k sensors; the
imaging pixels printed below total 3.09 billion) over a 9.6 deg² field of view, with
0.2″ pixels. *Source: Ivezić et al. 2019, §2.6.2, Fig. 12 and Table 1; LSST Science Book
2009, §2.4.*

**The numbers we draw with.** To place the CCDs we use the camera description in the Rubin
software, `lsst/obs_lsst` (commit `4cf6266c5d`, files `policy/rafts.yaml`,
`policy/cameraHeader.yaml` and `policy/lsstCam/R*.yaml`):

| quantity | value | source |
|---|---|---|
| raft spacing (center to center) | 127.0 mm | obs_lsst `rafts.yaml` |
| CCD spacing within a raft | 42.25 mm | obs_lsst `cameraHeader.yaml` |
| imaging pixels, e2v CCDs (13 rafts) | 4096 × 4004 | obs_lsst `cameraHeader.yaml` |
| imaging pixels, ITL CCDs (8 rafts) | 4072 × 4000 | obs_lsst `cameraHeader.yaml` |
| pixel size | 10 µm | obs_lsst; Ivezić et al. 2019 |
| plate scale | 20.006″ per mm, so 0.200″ per pixel | obs_lsst `cameraTransforms.yaml` |

Two vendors, Teledyne e2v and ITL (University of Arizona), built the CCDs, and their imaging
areas differ slightly (40.96 × 40.04 mm vs. 40.72 × 40.00 mm). Subtracting from the spacings,
the gaps between the imaging areas of neighboring CCDs in a raft are 1.29 mm and 2.21 mm
(e2v, in x and y) or 1.53 mm and 2.25 mm (ITL). Between rafts they are 0.25 mm wider. At
0.2″ per 10 µm pixel, 1 mm is 20″. These gaps are *our arithmetic* on the nominal layout;
they include the dead silicon at each CCD's edge, not just the air between packages.

**What we approximate.** This is the nominal layout: every CCD sits exactly on its grid
position, square to the axes. (The obs_lsst file itself notes that its e2v positions were
copied from the ITL ones and not checked against drawings or the sky.) We ignore the optical
distortion, which moves a star at the outer corners by about 0.4 mm (from obs_lsst's radial
distortion coefficients). The corner
sensors are drawn at their obs_lsst slots, without their small (≤ 5 mm) offsets. Ivezić et al.
2019 Table 1 gives a slightly different plate scale, 50.9 µm per arcsec, or 0.196″ per pixel.

In [ ]:
# LSSTCam geometry: nominal layout from lsst/obs_lsst (commit 4cf6266c5d), in mm on the focal plane.
PLATE_SCALE = 20.005867576692737   # arcsec per mm (obs_lsst cameraTransforms.yaml)
PIXEL_MM = 0.010                   # 10 um pixels
RAFT_PITCH_MM = 127.0              # raft center to raft center (obs_lsst rafts.yaml)
CCD_PITCH_MM = 42.25               # CCD center to CCD center within a raft (cameraHeader.yaml)
CCD_PIXELS = {"E2V": (4096, 4004), "ITL": (4072, 4000)}   # imaging pixels (x, y), per vendor
ITL_RAFTS = {"R01", "R02", "R03", "R10", "R20", "R41", "R42", "R43"}   # other science rafts: e2v
CORNER_RAFTS = {"R00": 180, "R04": 270, "R40": 90, "R44": 0}          # corner raft: rotation (deg)
MM_PER_DEG = 3600 / PLATE_SCALE    # focal-plane mm per degree on the sky (~180 mm)
FILL_FACTOR_PUBLISHED = 0.908      # Veres & Chesley 2017, AJ 154, 12, section 2.2

def science_ccds():
    """One row per science CCD: raft, sensor, vendor, center (x, y) and imaging size, in mm.

    Rafts are named R<row><column> and sensors S<row><column>, counting from the
    bottom left (-x, -y), as in obs_lsst.
    """
    rows = []
    for row in range(5):
        for col in range(5):
            raft = f"R{row}{col}"
            if raft in CORNER_RAFTS:
                continue
            vendor = "ITL" if raft in ITL_RAFTS else "E2V"
            nx, ny = CCD_PIXELS[vendor]
            for srow in range(3):
                for scol in range(3):
                    rows.append(dict(
                        raft=raft, sensor=f"S{srow}{scol}", vendor=vendor,
                        x=RAFT_PITCH_MM * (col - 2) + CCD_PITCH_MM * (scol - 1),
                        y=RAFT_PITCH_MM * (row - 2) + CCD_PITCH_MM * (srow - 1),
                        width=nx * PIXEL_MM, height=ny * PIXEL_MM))
    return pd.DataFrame(rows)

def corner_sensors():
    """Guide (SG0, SG1) and wavefront (SW0, SW1: half-size) sensors in the four corner rafts.

    Slot positions from obs_lsst cameraHeader.yaml, rotated with each corner raft;
    the few-mm per-sensor offsets are ignored.
    """
    rows = []
    slots = [("SG0", "guider", -CCD_PITCH_MM, 0.0, 40.72, 40.00),
             ("SG1", "guider", 0.0, -CCD_PITCH_MM, 40.00, 40.72),
             ("SW0", "wavefront", -CCD_PITCH_MM, -52.8125, 40.72, 20.00),
             ("SW1", "wavefront", -CCD_PITCH_MM, -31.6875, 40.72, 20.00)]
    for raft, yaw in CORNER_RAFTS.items():
        cx, cy = RAFT_PITCH_MM * (int(raft[2]) - 2), RAFT_PITCH_MM * (int(raft[1]) - 2)
        c, s = np.cos(np.radians(yaw)), np.sin(np.radians(yaw))
        for sensor, kind, dx, dy, w, h in slots:
            if yaw in (90, 270):
                w, h = h, w
            rows.append(dict(raft=raft, sensor=sensor, kind=kind, x=cx + c * dx - s * dy,
                             y=cy + s * dx + c * dy, width=w, height=h))
    return pd.DataFrame(rows)

def on_chip(x, y):
    """True where focal-plane position (x, y) in mm lands on the imaging area of a science CCD."""
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    col, row = np.rint(x / RAFT_PITCH_MM), np.rint(y / RAFT_PITCH_MM)     # raft index, -2..2
    ok = (np.abs(col) <= 2) & (np.abs(row) <= 2) & ~((np.abs(col) == 2) & (np.abs(row) == 2))
    u, v = x - RAFT_PITCH_MM * col, y - RAFT_PITCH_MM * row               # position in the raft
    scol, srow = np.rint(u / CCD_PITCH_MM), np.rint(v / CCD_PITCH_MM)     # CCD index, -1..1
    ok &= (np.abs(scol) <= 1) & (np.abs(srow) <= 1)
    itl = np.zeros(x.shape, dtype=bool)
    for raft in ITL_RAFTS:
        itl |= (row == int(raft[1]) - 2) & (col == int(raft[2]) - 2)
    half_x = np.where(itl, CCD_PIXELS["ITL"][0], CCD_PIXELS["E2V"][0]) * PIXEL_MM / 2
    half_y = np.where(itl, CCD_PIXELS["ITL"][1], CCD_PIXELS["E2V"][1]) * PIXEL_MM / 2
    return ok & (np.abs(u - CCD_PITCH_MM * scol) < half_x) & (np.abs(v - CCD_PITCH_MM * srow) < half_y)

def fill_factor():
    """Imaging area of the 189 science CCDs / area of the 21 raft cells (127 mm x 127 mm each)."""
    ccds = science_ccds()
    return float((ccds["width"] * ccds["height"]).sum() / (21 * RAFT_PITCH_MM ** 2))

def active_area_deg2():
    """Total imaging area of the science CCDs on the sky, deg^2."""
    ccds = science_ccds()
    return float((ccds["width"] * ccds["height"]).sum() / MM_PER_DEG ** 2)

ccds = science_ccds()
print(f"{len(ccds)} science CCDs in {ccds['raft'].nunique()} rafts:",
      ccds["vendor"].value_counts().to_dict())
print(f"pixels: {(ccds['width'] * ccds['height']).sum() / PIXEL_MM**2 / 1e9:.2f} billion, "
      f"{PIXEL_MM * PLATE_SCALE:.3f} arcsec each")
print(f"imaging area on the sky: {active_area_deg2():.2f} deg^2")
print(f"fill factor (this geometry): {fill_factor():.3f}   published: {FILL_FACTOR_PUBLISHED}")

In [ ]:
from matplotlib.patches import Rectangle, Circle

def draw_focal_plane(zoom=True, circle=True):
    """Draw the focal plane on the sky (degrees), with a zoom on the center of raft R22."""
    ncol = 2 if zoom else 1
    fig, axes = plt.subplots(1, ncol, figsize=(6 * ncol, 6.8))
    axes = np.atleast_1d(axes)
    colors = {"E2V": "tab:blue", "ITL": "tab:green", "guider": "tab:orange", "wavefront": "tab:red"}
    for ax in axes:
        for _, c in science_ccds().iterrows():
            ax.add_patch(Rectangle(((c.x - c.width / 2) / MM_PER_DEG, (c.y - c.height / 2) / MM_PER_DEG),
                                   c.width / MM_PER_DEG, c.height / MM_PER_DEG,
                                   color=colors[c.vendor], alpha=0.5, lw=0))
        for _, c in corner_sensors().iterrows():
            ax.add_patch(Rectangle(((c.x - c.width / 2) / MM_PER_DEG, (c.y - c.height / 2) / MM_PER_DEG),
                                   c.width / MM_PER_DEG, c.height / MM_PER_DEG,
                                   color=colors[c.kind], alpha=0.6, lw=0))
        for row in range(5):                          # raft outlines
            for col in range(5):
                if f"R{row}{col}" in CORNER_RAFTS:
                    continue
                x0 = (RAFT_PITCH_MM * (col - 2) - RAFT_PITCH_MM / 2) / MM_PER_DEG
                y0 = (RAFT_PITCH_MM * (row - 2) - RAFT_PITCH_MM / 2) / MM_PER_DEG
                ax.add_patch(Rectangle((x0, y0), RAFT_PITCH_MM / MM_PER_DEG, RAFT_PITCH_MM / MM_PER_DEG,
                                       fill=False, ec="k", lw=0.6))
        if circle:
            ax.add_patch(Circle((0, 0), 1.75, fill=False, ec="k", ls="--", lw=1.2))
        ax.set_aspect("equal")
        ax.set_xlabel("x on the sky (deg)")
        ax.set_ylabel("y on the sky (deg)")
    ax = axes[0]
    ax.set_xlim(-2.1, 2.1)
    ax.set_ylim(-2.1, 2.1)
    handles = [Rectangle((0, 0), 1, 1, color=colors[k], alpha=0.5) for k in colors]
    labels = ["e2v science CCD", "ITL science CCD", "guide sensor", "wavefront sensor"]
    if circle:
        handles.append(Circle((0, 0), 1, fill=False, ec="k", ls="--"))
        labels.append("Notebook A's 1.75° circle")
    ax.legend(handles, labels, fontsize=8, loc="upper center", ncol=3, bbox_to_anchor=(0.5, -0.1))
    ax.set_title("LSSTCam focal plane (nominal obs_lsst layout)")
    if zoom:
        axes[1].set_xlim(-0.2, 0.2)
        axes[1].set_ylim(-0.2, 0.2)
        axes[1].set_title("zoom: center of raft R22 (gaps are real)")
    plt.tight_layout()
    plt.show()

draw_focal_plane()

### Fill factor

The *fill factor* is the fraction of the focal plane covered by pixels. From the drawing above
we compute it as the imaging area of the 189 science CCDs divided by the area of the 21 raft
cells (127 mm on a side):

$$f_\text{fill} = \frac{\sum_\text{CCDs} (\text{imaging width} \times \text{height})}{21 \times (127\ \text{mm})^2}$$

This gives ≈ 0.913 (from our geometry). The published values:

- **90.8%**: "The current LSST camera design expects a 90.8% fill factor."
  *Vereš & Chesley 2017, AJ 154, 12, §2.2* (their near-Earth-object simulations).
- **93%**: "gaps of less than a few hundred µm. The resulting 'fill factor,' i.e., the fraction
  of the focal plane covered by pixels, is 93%." *LSST Science Book 2009, §2.4.2.*
- **> 90%**, "including non-imaging silicon area and inter-chip gaps." *O'Connor et al. 2019,
  JATIS 5, 041508, §1.*

**Where does this break?** The answer depends on the denominator. What is "the focal plane":
the 21 raft cells, the 9.6 deg² circle, or the square that encloses the science rafts? The
Science Book's gaps are "less than a few hundred µm," but ours are 1.3–2.3 mm. Which gap is
each number about, and which one matters for your science?

### What Notebook A's circle hid

In Notebook A every visit was a circle of radius 1.75° (9.62 deg²), and the chip gaps were
ignored. The drawing shows the dashed circle on the real layout. The next cell measures on a
fine grid how much of that circle is actually on silicon, and how much silicon sticks out past
it (the corners of the outer rafts).

In [ ]:
def circle_vs_silicon(radius_deg=1.75, step_deg=0.004):
    """Compare the 1.75-deg circle of Notebook A with the real CCD layout, on a fine grid.

    Returns the fraction of the circle that lands on a science CCD, and the fraction of all
    science-CCD area that lies outside the circle.
    """
    g = np.arange(-2.2, 2.2, step_deg) + step_deg / 2
    X, Y = np.meshgrid(g, g)
    on = on_chip(X * MM_PER_DEG, Y * MM_PER_DEG)
    inside = X**2 + Y**2 < radius_deg**2
    return {"circle_on_silicon": float(on[inside].mean()),
            "silicon_outside_circle": float(on[~inside].sum() / on.sum())}

cvs = circle_vs_silicon()
print(f"circle of 1.75 deg: {np.pi * 1.75**2:.2f} deg^2; science CCDs: {active_area_deg2():.2f} deg^2")
print(f"fraction of the circle on silicon:      {cvs['circle_on_silicon']:.3f}")
print(f"fraction of the silicon outside circle: {cvs['silicon_outside_circle']:.3f}")

**Same area, different shape.** The circle and the silicon cover almost the same area, so
Notebook A's visit counts are right *on average*. What does the circle get wrong for a single
object near the edge of a field, or for a transient that falls in a gap?

### Dithering

If the telescope pointed at exactly the same spot, with the camera at the same angle, on every
visit, the same stars would land in the same gaps every time. They would never be observed. The
fix is to *dither*: shift the pointing (and rotate the camera) a little from visit to visit, so
the gaps move around on the sky.

The next cell takes N visits of one field. *Without dithering*, every visit has the same
pointing and the same rotation. *With dithering*, each visit is offset in a random direction by
up to `dither_deg` (uniform over a disk), and, if `rotate=True`, turned by a random angle between
−90° and +90°. The default, 0.7°, is about one raft width (0.71°). These choices are ours, not
the survey's. The maps count visits on a grid of 0.006° (22″) pixels, finer than the narrowest
gap (1.29 mm = 26″). The histograms use only the central 1° × 1° box (the white square). Even
with the largest dither on the slider, that box always stays inside the outline of the science
rafts, so every zero in it is a gap, not the edge of the field. The random numbers are seeded,
so a given `seed` always gives the same answer.

If the slider doesn't appear, call `show_dithering(n_visits, dither_deg, rotate, seed)`
directly.

In [ ]:
def visit_counts(n_visits=20, dither_deg=0.7, rotate=True, seed=367, half_deg=2.4, step_deg=0.006):
    """Number of visits that put each sky pixel on a science CCD.

    Returns (counts, grid): counts is a 2D array over the square |x|, |y| < half_deg (deg),
    grid the pixel centers along each axis. dither_deg = 0 and rotate = False: no dithering.
    """
    rng = np.random.default_rng(seed)
    grid = np.arange(-half_deg, half_deg, step_deg) + step_deg / 2
    X, Y = np.meshgrid(grid, grid)
    counts = np.zeros(X.shape, dtype=int)
    for _ in range(n_visits):
        r = dither_deg * np.sqrt(rng.uniform())            # uniform over a disk of radius dither_deg
        phi = rng.uniform(0, 2 * np.pi)
        theta = np.radians(rng.uniform(-90, 90)) if rotate else 0.0
        dx, dy = X - r * np.cos(phi), Y - r * np.sin(phi)  # sky offset from this visit's pointing
        c, s = np.cos(theta), np.sin(theta)
        counts += on_chip((c * dx + s * dy) * MM_PER_DEG, (-s * dx + c * dy) * MM_PER_DEG)
    return counts, grid

BOX_DEG = 0.5    # half-width of the central box used for the histograms

def _box_stats(counts, grid):
    box = np.abs(grid) < BOX_DEG
    sub = counts[np.ix_(box, box)]
    return {"frac_zero": float(np.mean(sub == 0)), "mean": float(sub.mean()),
            "std": float(sub.std()), "min": int(sub.min()), "max": int(sub.max())}

def show_dithering(n_visits=20, dither_deg=0.7, rotate=True, seed=367):
    """Maps of visit counts without and with dithering, and their histograms in the central box.

    Returns {"undithered": stats, "dithered": stats}; stats are the fraction of pixels never
    observed, and the mean, standard deviation, min and max of the visit count in the box.
    """
    runs = {"undithered": visit_counts(n_visits, 0.0, False, seed),
            "dithered": visit_counts(n_visits, dither_deg, rotate, seed)}
    stats = {k: _box_stats(*v) for k, v in runs.items()}
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
    bins = np.arange(-0.5, n_visits + 1.5)
    for ax, (name, (counts, grid)) in zip(axes, runs.items()):
        ext = [grid[0], grid[-1], grid[0], grid[-1]]
        im = ax.imshow(counts, origin="lower", extent=ext, cmap="viridis", vmin=0, vmax=n_visits,
                       interpolation="nearest")
        ax.add_patch(Rectangle((-BOX_DEG, -BOX_DEG), 2 * BOX_DEG, 2 * BOX_DEG, fill=False, ec="w", lw=1.2))
        title = "no dithering" if name == "undithered" else (
            f"dithered: up to {dither_deg}°" + (", rotated" if rotate else ""))
        ax.set_title(f"{title}\n{n_visits} visits; {stats[name]['frac_zero']:.1%} of box never seen")
        ax.set_xlabel("x (deg)")
        ax.set_ylabel("y (deg)")
        plt.colorbar(im, ax=ax, label="visits", shrink=0.85)
        box = np.abs(grid) < BOX_DEG
        axes[2].hist(counts[np.ix_(box, box)].ravel(), bins=bins, histtype="step", lw=1.5,
                     density=True, label=title)
    axes[2].set_xlabel("visits per pixel (central box)")
    axes[2].set_ylabel("fraction of pixels")
    axes[2].set_yscale("log")
    axes[2].legend(fontsize=8, loc="upper left")
    plt.tight_layout()
    plt.show()
    for name, s in stats.items():
        print(f"{name:11}: never seen {s['frac_zero']:6.1%}   visits mean {s['mean']:5.2f}  "
              f"std {s['std']:5.2f}  min {s['min']}  max {s['max']}")
    return stats

_ = show_dithering(20)

In [ ]:
from ipywidgets import Checkbox, FloatSlider, IntSlider

def _dither_widget(n_visits, dither_deg, rotate):
    show_dithering(n_visits, dither_deg, rotate)       # don't echo the returned stats

sliders(_dither_widget,
        n_visits=IntSlider(value=20, min=1, max=50, step=1, description="visits", **_W),
        dither_deg=FloatSlider(value=0.7, min=0.0, max=0.75, step=0.05,
                               description="dither radius (deg)", **_W),
        rotate=Checkbox(value=True, description="rotate the camera"));

**Depth uniformity.** With the default dithering, essentially no pixel in the box is missed, but the pixels no longer
all get the same number of visits. Monday's worksheet says a coadd of N visits gains
1.25 log₁₀ N in depth. Use the histogram: how much does the coadded depth vary across the box,
with and without dithering? Which is worse for your science, a few pixels that are never
observed or every pixel a little different?

**The selection function.** The *selection function* is the probability that an object at a
given position (and brightness) makes it into your catalog. With dithering, what does it look
like on the sky, and on what angular scales does it vary? Why would a weak-lensing or
galaxy-clustering measurement care about a pattern with the spacing of the CCDs or the rafts?

**Where does this break?** This is one field, alone. The real survey tiles the sky with
overlapping fields, so the edges of one field are covered by its neighbors. How would the
overlaps change the histogram? The survey also chooses *when* to dither (per visit or per
night) and how far. Try `dither_deg=0.1` or `rotate=False`. What survives?

> **Go deeper.** The `camera_footprint_demo` notebook in rubin_sim_notebooks
> (<https://github.com/lsst/rubin_sim_notebooks/blob/main/utils/camera_footprint_demo.ipynb>)
> uses rubin_sim's own map of which sky positions land on silicon, built from the same
> obs_lsst camera model. How big an error do the chip gaps make in the number of visits a
> survey simulation reports for a single star?

## Part 4 · Go deeper (optional)

- **The throughput model itself.** The notebooks in `lsst-pst/syseng_throughputs`
  (<https://github.com/lsst-pst/syseng_throughputs/tree/main/notebooks>) show how the curves
  in Part 1 are built. `SilverVsAluminum.ipynb` compares mirror coatings: release 1.9, the
  one used here, moved the mirrors from Al-Ag-Al (aluminum, silver, aluminum) to silver on all
  three (*syseng_throughputs README*). Which bands gained depth and which lost it?
- **Measured seeing and depth.** On the Rubin Science Platform (once your accounts are live),
  the DP1 Visit table tutorial, `201_10_Visit_table.ipynb`
  (<https://github.com/lsst/tutorial-notebooks/tree/main/DP1/200_Data_Products/201_Catalogs>),
  gives the measured seeing and depth of the LSSTComCam commissioning visits in DP1. How do they
  compare with your `m5()`? (LSSTComCam is a smaller commissioning camera, so the field of
  view is different from LSSTCam's.)
- **The camera, from the people who built it.** Aaron Roodman's DESC Dark Energy School talk
  "The LSST Camera: design drivers and expected performance" (DE School XIII, SLAC, July 2023)
  is linked, with slides and video, from <https://lsstdesc.org/pages/DESchool.html>.
- **The footprint in rubin_sim.** `camera_footprint_demo.ipynb`
  (<https://github.com/lsst/rubin_sim_notebooks/blob/main/utils/camera_footprint_demo.ipynb>)
  shows which points on the sky fall on silicon for a given pointing and rotation.
- **Simulating images.** GalSim (<https://github.com/GalSim-developers/GalSim>; Rowe et al.
  2015) is a widely used package for simulating astronomical images of galaxies and stars. Draw a
  galaxy, convolve it with a 0.7″ and a 1.1″ PSF, and add sky noise: does the Part 2 toy
  model's resolution cut look right?

## Hand-in

Due Monday Oct 5 on Canvas. Turn in: (1) one figure you made (use a direct function call, not
a slider); (2) one paragraph: what does your team's science case need from the camera and
telescope, and does the as-built system give it?; (3) a sentence on any AI tools you used.
Upload the .ipynb or a PDF. Graded complete/incomplete.

Make your figure in the code cell below. Write your paragraph and the AI-tools sentence in the
last cell.

In [ ]:
# Your hand-in figure (use a direct function call, not a slider), e.g.
# _ = show_dithering(n_visits=30, dither_deg=0.3, rotate=False)

*Your paragraph here.*